In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 2: Train model
# =============================================================================
# Step:         2 of 5
# Summary:      Build and train the BiLSTM + Bahdanau-attention servingCell forecaster on Step 1's windowed arrays.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 2 — Train model

Part of the **FUMD-AI training & postprocessing workflow** (step 2 of 5).

**Purpose.** Train the multi-step `servingCell` forecaster
(`fumd_training_workflow.model.build_model`) on the scaled sliding-window
arrays Step 1 produced: a 2x bidirectional-LSTM encoder feeding a per-step
Bahdanau-attention decoder that autoregressively emits one softmax
classification per forecast step. Same architecture as the original
exploratory notebook
(`combined_TimeSequence_*_BahdanauAtention_encoder_2BLSTM.ipynb`) -- a
validated design -- rebuilt as one parameterized, importable model function
instead of one-off inline cells.

**Input:** `INPUT_DIR` -- Step 1's output directory (`windows.npz`,
`run_manifest.json`).

**Outputs**, written under `OUTPUT_DIR` (defaults to the same directory as
`INPUT_DIR`, so Step 1+2's artifacts live together for Step 3):
- `model.keras` -- the trained model
- `training_history.json` -- per-epoch loss/accuracy for every output head
- `training_curves.png` -- loss and accuracy vs. epoch, saved (not just
  shown) so this also works in a non-interactive Slurm batch job
- `run_manifest.json` -- Step 1's manifest with model hyperparameters
  appended

**GPU.** This step is the one that actually needs a GPU -- see
`slurm/train_model.sbatch` for the Slurm/Singularity submission this
notebook is designed to run under (`jupyter execute
notebooks/step_2_train_model.ipynb`, no papermill/parameter-injection tool
required -- see `src/fumd_training_workflow/notebook_runner.py`'s
docstring for why).


In [ ]:
import json
import os
import sys

import joblib
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")
from fumd_training_workflow.model import build_model
from fumd_training_workflow.metrics_io import load_run_manifest, save_run_manifest

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# ---- Parameters ----
INPUT_DIR = "pipeline_run"   # Step 1's output directory
OUTPUT_DIR = "pipeline_run"  # where model.keras / training_history.json / run_manifest.json are (re)written

LSTM_UNITS = 128
ATTENTION_UNITS = 128
EMBEDDING_DIM = 128
DENSE_UNITS = 128
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001

EPOCHS = 50
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.2         # carved out of the *training* windows, for early stopping only
EARLY_STOPPING_PATIENCE = 3
RANDOM_SEED = 42

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute notebooks/step_2_train_model.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

INPUT_DIR = env_override('INPUT_DIR', INPUT_DIR)
OUTPUT_DIR = env_override('OUTPUT_DIR', OUTPUT_DIR)
LSTM_UNITS = env_override('LSTM_UNITS', LSTM_UNITS)
ATTENTION_UNITS = env_override('ATTENTION_UNITS', ATTENTION_UNITS)
EMBEDDING_DIM = env_override('EMBEDDING_DIM', EMBEDDING_DIM)
DENSE_UNITS = env_override('DENSE_UNITS', DENSE_UNITS)
DROPOUT_RATE = env_override('DROPOUT_RATE', DROPOUT_RATE)
LEARNING_RATE = env_override('LEARNING_RATE', LEARNING_RATE)
EPOCHS = env_override('EPOCHS', EPOCHS)
BATCH_SIZE = env_override('BATCH_SIZE', BATCH_SIZE)
VALIDATION_SPLIT = env_override('VALIDATION_SPLIT', VALIDATION_SPLIT)
EARLY_STOPPING_PATIENCE = env_override('EARLY_STOPPING_PATIENCE', EARLY_STOPPING_PATIENCE)
RANDOM_SEED = env_override('RANDOM_SEED', RANDOM_SEED)

## 1. Load Step 1's windows and manifest

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

manifest = load_run_manifest(INPUT_DIR)
data_npz = np.load(os.path.join(INPUT_DIR, "windows.npz"))
X_train, y_train = data_npz["X_train"], data_npz["y_train"]
X_val, y_val = data_npz["X_val"], data_npz["y_val"]

sequence_length = manifest["sequence_length"]
future_steps = manifest["future_steps"]
num_base_stations = manifest["num_base_stations"]
num_features = X_train.shape[-1]

print(f"X_train {X_train.shape}, y_train {y_train.shape}")
print(f"num_features={num_features}, num_base_stations={num_base_stations}, future_steps={future_steps}")

## 2. Build the model

In [ ]:
model = build_model(
    sequence_length=sequence_length,
    num_features=num_features,
    future_steps=future_steps,
    num_base_stations=num_base_stations,
    lstm_units=LSTM_UNITS,
    attention_units=ATTENTION_UNITS,
    embedding_dim=EMBEDDING_DIM,
    dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE,
    learning_rate=LEARNING_RATE,
)
model.summary()

## 3. Train

`sparse_categorical_crossentropy` expects one label array per output head -- `y_train[:, k]` is the target `future_steps[k]` seconds ahead (see `data.build_sequences`'s docstring).

In [ ]:
y_train_per_output = [y_train[:, k] for k in range(len(future_steps))]

early_stopping = EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

history = model.fit(
    X_train, y_train_per_output,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    callbacks=[early_stopping],
)

## 4. Save

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_path = os.path.join(OUTPUT_DIR, "model.keras")
model.save(model_path)

history_path = os.path.join(OUTPUT_DIR, "training_history.json")
with open(history_path, "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

manifest.update({
    "lstm_units": LSTM_UNITS,
    "attention_units": ATTENTION_UNITS,
    "embedding_dim": EMBEDDING_DIM,
    "dense_units": DENSE_UNITS,
    "dropout_rate": DROPOUT_RATE,
    "learning_rate": LEARNING_RATE,
    "epochs_requested": EPOCHS,
    "epochs_run": len(history.history["loss"]),
    "batch_size": BATCH_SIZE,
    "random_seed": RANDOM_SEED,
})
save_run_manifest(OUTPUT_DIR, manifest)

print(f"saved {model_path}, {history_path}, run_manifest.json -> {OUTPUT_DIR}")

## 5. Training curves

In [ ]:
import matplotlib
matplotlib.use("Agg")  # never blocks / requires a display -- safe under `jupyter execute` and Slurm batch jobs
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Training Loss")
axes[0].plot(history.history["val_loss"], label="Validation Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[0].set_title("Overall loss")

for k, step in enumerate(future_steps):
    axes[1].plot(history.history[f"output_{k + 1}_accuracy"], label=f"+{step}s train", alpha=0.6)
    axes[1].plot(history.history[f"val_output_{k + 1}_accuracy"], linestyle="--", label=f"+{step}s val", alpha=0.6)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
axes[1].set_title("Per-output accuracy")

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
print(f"saved training_curves.png -> {OUTPUT_DIR}")